# Portafolio: análisis conjunto de las dos patas abiertas

**Proyecto Final · Productos Derivados y Administración de Riesgos Corporativos**  
EGADE Business School · Equipo 6 · Grupo 40  
Eduardo Ayala Saldaña, Mariann Antonelli Morales Tobias, Ulises Salinas López  
Dr. Raymundo Díaz Robles · Noviembre de 2026

Origen: `06_codigo_python/cartera.py`

```
Analisis conjunto de la cartera al 12-nov-2026.
Pata 1: call 6CZ6 20-nov K=0.715, 10 contratos, comprada en 0.0026.
Pata 2: put  ZNZ6 20-nov K=107.25, 10 contratos, al offer de 0.546875.

El objetivo no es sumar dos P&L sino medir si las patas se compensan.
```

In [1]:
from math import log, sqrt, exp, erf, pi

RF, T0, T1 = 0.0394, 15.32/365.0, 8.32/365.0
N = lambda x: 0.5*(1.0+erf(x/sqrt(2.0)))

In [2]:
def b76(F,K,T,r,s,tipo):
    if s<=0 or T<=0: return exp(-r*T)*(max(F-K,0.0) if tipo=="c" else max(K-F,0.0))
    d1=(log(F/K)+0.5*s*s*T)/(s*sqrt(T)); d2=d1-s*sqrt(T)
    if tipo=="c": return exp(-r*T)*(F*N(d1)-K*N(d2))
    return exp(-r*T)*(K*N(-d2)-F*N(-d1))

In [3]:
# ---- pata CAD ----
FC, KC, SC, NC, PC, MC = 0.7145, 0.715, 0.0453, 10, 0.0026, 100_000.0
# ---- pata ZN ----
FZ, KZ, SZ, NZ, PZ, MZ = 107.3172, 107.25, 0.0663, 10, 0.546875, 1_000.0

costo_c, costo_z = PC*MC*NC, PZ*MZ*NZ
total = costo_c + costo_z

In [4]:
print("="*78)
print("A. LA CARTERA")

A. LA CARTERA


In [5]:
print("="*78)
print(f"   Call 6CZ6  0.715 x{NC:2d} @ {PC:<9}  {costo_c:>9,.0f} USD")
print(f"   Put  ZNZ6 107.25 x{NZ:2d} @ {PZ:<9}  {costo_z:>9,.0f} USD")
print(f"   {'':48}{'-'*12}")
print(f"   Prima total comprometida{'':24}{total:>9,.0f} USD")
print(f"   Sobre capital de 90,162.50{'':22}{total/90162.50*100:>8.2f} %")
print(f"   Perdida maxima posible{'':26}{-total:>9,.0f} USD")

print()

   Call 6CZ6  0.715 x10 @ 0.0026         2,600 USD
   Put  ZNZ6 107.25 x10 @ 0.546875       5,469 USD
                                                   ------------
   Prima total comprometida                            8,069 USD
   Sobre capital de 90,162.50                          8.95 %
   Perdida maxima posible                             -8,069 USD



In [6]:
print("="*78)
print("B. MATRIZ CONJUNTA DE P&L AL 12-NOV")

B. MATRIZ CONJUNTA DE P&L AL 12-NOV


In [7]:
print("="*78)
print("   Filas: movimiento del dolar canadiense.")
print("   Columnas: movimiento del precio del bono (1 punto ~ 13 pb de")
print("   rendimiento; aproximacion con duracion modificada de 7.5 anos,")
print("   la cifra exacta exige identificar el bono entregable mas barato).\n")

movz = [-1.5,-0.75,0.0,0.75,1.5]      # puntos de precio del futuro ZN
movc = [-0.015,-0.0075,0.0,0.0075,0.015]

print(f"   {'':>10}", end="")
for mz in movz:
    print(f"{('bono '+f'{mz:+.2f}'):>13}", end="")
print()
print(f"   {'':>10}", end="")
for mz in movz:
    print(f"{('~'+f'{-mz*13:+.0f}'+' pb'):>13}", end="")
print("\n")

mejor, peor = None, None
for mc in movc:
    F1 = FC*(1+mc)
    vc = b76(F1,KC,T1,RF,SC,"c")*MC*NC
    print(f"   CAD {mc*100:+5.2f}%", end="")
    for mz in movz:
        F2 = FZ + mz
        vz = b76(F2,KZ,T1,RF,SZ,"p")*MZ*NZ
        pnl = vc + vz - total
        if mejor is None or pnl > mejor[0]: mejor = (pnl, mc, mz)
        if peor  is None or pnl < peor[0]:  peor  = (pnl, mc, mz)
        print(f"{pnl:>+12,.0f}$", end="")
    print()

print(f"\n   Mejor celda: {mejor[0]:+,.0f} USD  (CAD {mejor[1]*100:+.2f}%, bono {mejor[2]:+.2f})")
print(f"   Peor  celda: {peor[0]:+,.0f} USD  (CAD {peor[1]*100:+.2f}%, bono {peor[2]:+.2f})")

print()

   Filas: movimiento del dolar canadiense.
   Columnas: movimiento del precio del bono (1 punto ~ 13 pb de
   rendimiento; aproximacion con duracion modificada de 7.5 anos,
   la cifra exacta exige identificar el bono entregable mas barato).

                bono -1.50   bono -0.75   bono +0.00   bono +0.75   bono +1.50
                   ~+20 pb      ~+10 pb       ~-0 pb      ~-10 pb      ~-20 pb

   CAD -1.50%      +6,705$        +465$      -4,099$      -6,660$      -7,697$
   CAD -0.75%      +6,959$        +719$      -3,844$      -6,405$      -7,442$
   CAD +0.00%      +8,397$      +2,156$      -2,407$      -4,968$      -6,005$
   CAD +0.75%     +11,959$      +5,718$      +1,155$      -1,406$      -2,443$
   CAD +1.50%     +16,931$     +10,690$      +6,127$      +3,566$      +2,529$

   Mejor celda: +16,931 USD  (CAD +1.50%, bono -1.50)
   Peor  celda: -7,697 USD  (CAD -1.50%, bono +1.50)



In [8]:
print("="*78)
print("C. LA PATA DE BONOS COMO DIVERSIFICADOR")

C. LA PATA DE BONOS COMO DIVERSIFICADOR


In [9]:
print("="*78)
print("   Las esquinas de la matriz identifican los cuatro regimenes:\n")
reg = [
 ("Fed restrictiva: tasas suben, dolar se fortalece", +1, -1),
 ("Reflacion: tasas suben, dolar se debilita",        +1, +1),
 ("Fed acomodaticia: tasas bajan, dolar se debilita", -1, +1),
 ("Aversion al riesgo: tasas bajan, dolar se fortalece", -1, -1),
]
for nom, sz, sc in reg:
    F1 = FC*(1 + sc*0.015); F2 = FZ - sz*1.5
    vc = b76(F1,KC,T1,RF,SC,"c")*MC*NC
    vz = b76(F2,KZ,T1,RF,SZ,"p")*MZ*NZ
    print(f"   {nom:<52}")
    print(f"      CAD {vc-costo_c:>+9,.0f}$   ZN {vz-costo_z:>+9,.0f}$   "
          f"total {vc+vz-total:>+9,.0f}$")
print("\n   Solo el cuarto regimen castiga a las dos patas a la vez, y es el")
print("   menos probable de los cuatro: exige que el dolar se fortalezca")
print("   mientras los rendimientos caen, combinacion propia de una fuga")
print("   hacia la calidad. En los dos regimenes centrales una pata")
print("   financia a la otra.")

   Las esquinas de la matriz identifican los cuatro regimenes:

   Fed restrictiva: tasas suben, dolar se fortalece    
      CAD    -2,583$   ZN    +9,288$   total    +6,705$
   Reflacion: tasas suben, dolar se debilita           
      CAD    +7,642$   ZN    +9,288$   total   +16,931$
   Fed acomodaticia: tasas bajan, dolar se debilita    
      CAD    +7,642$   ZN    -5,114$   total    +2,529$
   Aversion al riesgo: tasas bajan, dolar se fortalece 
      CAD    -2,583$   ZN    -5,114$   total    -7,697$

   Solo el cuarto regimen castiga a las dos patas a la vez, y es el
   menos probable de los cuatro: exige que el dolar se fortalezca
   mientras los rendimientos caen, combinacion propia de una fuga
   hacia la calidad. En los dos regimenes centrales una pata
   financia a la otra.
